# 🎯 Clase 5 — Regresión Logística y Scoring Crediticio (R)

### Aplicaciones de Minería de Datos a Economía y Finanzas
**Maestría en Minería de Datos · UTN Facultad Regional Rosario · 2026**

*Dr. Darío Ezequiel Díaz · Jueves 7 de mayo de 2026*

---

## 📋 Objetivos de la práctica en R

Al finalizar este notebook, ustedes serán capaces de:

1. **Ajustar** un modelo de regresión logística sobre el German Credit Dataset con `stats::glm()` por máxima verosimilitud pura
2. **Interpretar** los coeficientes mediante odds ratios e intervalos de confianza al 95 %
3. **Construir** una scorecard completa con el paquete `scorecard` de Shichen Xie usando el flujo industrial: binning supervisado → WoE → glm → scorecard
4. **Evaluar** la performance con AUC-ROC, KS y Gini usando `perf_eva()` del mismo paquete
5. **Comparar** los resultados con el notebook de Python sobre el mismo dataset

## 🗺️ Mapa del notebook

| Sección | Contenido |
|---------|-----------|
| 1 | Configuración del entorno R en Colab |
| 2 | Carga del dataset German Credit |
| 3 | Exploración inicial con tidyverse |
| 4 | Diagnóstico de cardinalidad y agrupamiento preventivo |
| 5 | Ajuste con `glm()` base (MV pura) |
| 6 | Interpretación de coeficientes y odds ratios |
| 7 | Flujo industrial completo con paquete `scorecard` |
| 8 | Métricas de evaluación con `perf_eva()` |
| 9 | Diagnóstico de supuestos (linealidad logit, VIF) |
| 10 | Síntesis y comparación R vs Python |

---

## 🔄 Diferencias filosóficas R vs Python

| Aspecto | Python (sklearn) | R (`scorecard`) |
|---------|------------------|-----------------|
| Regularización default | **L2 (C=1.0)** silenciosa | **Sin regularización** (MV pura) |
| Discretización WoE | Manual con `optbinning` | **Integrada y automática** |
| Inferencia clásica | `statsmodels` separado | **`glm()` lo incluye** |
| Salida `summary()` | Estilo machine learning | **Estilo SAS/regulatorio** |
| Construcción scorecard | Manual con `prob_a_score()` | **Función `scorecard()` lista** |

> 💡 R está históricamente más alineado con la práctica regulatoria bancaria (SAS-like). Por eso muchas entidades financieras argentinas mantienen sus modelos en R o SAS.

---

> 📌 **Recomendación:** ejecuten las celdas en orden secuencial. La sección 1 carga el setup script personalizado del curso con ~237 paquetes pre-instalados.


## 1. Configuración del entorno R en Colab

Cargamos el setup script del curso (Drive personal), que contiene aproximadamente 237 paquetes precompilados para R en Colab. Este script ahorra unos 8-10 minutos de instalaciones individuales por sesión.

### Paquetes que usaremos en esta clase

- **`tidyverse`** (dplyr, ggplot2, tibble): manipulación y visualización
- **`scorecard`**: flujo industrial de credit scoring (Xie, 2020)
- **`pROC`**: curva ROC y AUC
- **`car`**: cálculo de VIF
- **`broom`**: tidy output de `glm()`
- **`gtsummary`**: tablas profesionales

> ⚠️ Si trabajan en Colab sin acceso al setup script, las instrucciones de instalación manual están al final de esta sección.


In [ ]:
# Cargar setup script del curso (ruta personal en Drive)
# Si no está disponible, descomentar el bloque de instalación manual abajo

ruta_setup <- "/content/drive/MyDrive/R_Colab/setup_R_colab.R"

if (file.exists(ruta_setup)) {
  source(ruta_setup)
  cat("✅ Setup script cargado desde Drive\n")
  cat("   ~237 paquetes pre-instalados disponibles\n")
} else {
  cat("⚠️  Setup script no disponible. Instalación manual:\n\n")
  cat("install.packages(c('tidyverse', 'scorecard', 'pROC',\n")
  cat("                   'car', 'broom', 'gtsummary'),\n")
  cat("                 repos = 'https://cran.rstudio.com/',\n")
  cat("                 dependencies = TRUE)\n")
}


In [ ]:
# Algunos paquetes pueden faltar en el setup; los instalamos puntualmente
paquetes_clase5 <- c("scorecard", "pROC", "car", "broom", "gtsummary")

for (pkg in paquetes_clase5) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    cat(sprintf("📦 Instalando %s...\n", pkg))
    install.packages(pkg, repos = "https://cran.rstudio.com/", quiet = TRUE)
  }
}
cat("\n✅ Todos los paquetes de Clase 5 disponibles\n")


In [ ]:
# Carga de librerías
suppressPackageStartupMessages({
  library(tidyverse)    # manipulación + ggplot2
  library(scorecard)    # paquete central de la práctica
  library(pROC)         # curvas ROC y AUC
  library(car)          # VIF
  library(broom)        # tidy() para glm
})

# Configuración global del curso
RNG_SEED <- 2026
set.seed(RNG_SEED)

# ═══ Paleta cromática Unidad 3: Royal Sapphire & Gold ═══
COLOR_INDIGO   <- "#1E1B4B"
COLOR_VIOLET   <- "#6D28D9"
COLOR_GOLD     <- "#D97706"
COLOR_EMERALD  <- "#059669"
COLOR_CARMIN   <- "#DC2626"
COLOR_GRAY     <- "#64748B"

# Tema ggplot2 personalizado
tema_unidad3 <- theme_minimal(base_size = 11) +
  theme(
    plot.title = element_text(size = 13, face = "bold", color = COLOR_INDIGO),
    plot.subtitle = element_text(size = 10, color = COLOR_GRAY),
    axis.title = element_text(face = "bold"),
    panel.grid.minor = element_blank(),
    legend.position = "right"
  )

theme_set(tema_unidad3)

cat(sprintf("🎯 RNG_SEED = %d\n", RNG_SEED))
cat("🎨 Paleta cromática Unidad 3 cargada\n")
cat("📚 Librerías importadas: tidyverse, scorecard, pROC, car, broom\n")


## 2. Carga del German Credit Dataset

El paquete `scorecard` incluye internamente el dataset **`germancredit`** ya cargado, lo que simplifica la carga respecto del flujo de Python con OpenML.

### Estrategia de carga resiliente

1. **Primer intento**: `scorecard::germancredit` (dataset interno del paquete)
2. **Segundo intento**: URL pública de OpenML
3. **Tercer intento**: CSV local en Drive


In [ ]:
# Función de carga resiliente
cargar_german_credit <- function() {

  # Intento 1: dataset interno del paquete scorecard
  resultado <- try({
    data(germancredit, package = "scorecard", envir = environment())
    germancredit
  }, silent = TRUE)

  if (!inherits(resultado, "try-error")) {
    cat(sprintf("✅ Dataset cargado desde scorecard::germancredit: %d × %d\n",
                nrow(resultado), ncol(resultado)))
    return(resultado)
  }

  # Intento 2: URL pública
  url_openml <- "https://www.openml.org/data/get_csv/31/dataset_31_credit-g.arff"
  resultado <- try(read.csv(url_openml), silent = TRUE)
  if (!inherits(resultado, "try-error")) {
    cat(sprintf("✅ Dataset cargado desde URL pública: %d × %d\n",
                nrow(resultado), ncol(resultado)))
    return(resultado)
  }

  # Intento 3: Drive
  ruta_drive <- "/content/drive/MyDrive/datasets/german_credit.csv"
  resultado <- try(read.csv(ruta_drive), silent = TRUE)
  if (!inherits(resultado, "try-error")) {
    cat(sprintf("✅ Dataset cargado desde Drive: %d × %d\n",
                nrow(resultado), ncol(resultado)))
    return(resultado)
  }

  stop("❌ No se pudo cargar el dataset por ninguna vía.")
}

df <- cargar_german_credit()
head(df)


In [ ]:
# Recodificación de la variable respuesta según convención regulatoria
# Y = 1 para evento adverso (default), Y = 0 para pago regular
# El dataset scorecard::germancredit trae 'creditability' como factor con niveles 'good'/'bad'

df$Y <- as.integer(df$creditability == "bad")

# Verificamos la distribución
cat("📊 Distribución de la variable respuesta:\n\n")
tabla_Y <- table(Y = df$Y)
print(tabla_Y)

cat(sprintf("\n   Tasa de default observada: %.2f%%\n", mean(df$Y) * 100))
cat(sprintf("   Tasa de pago regular:      %.2f%%\n", (1 - mean(df$Y)) * 100))


## 3. Exploración inicial con tidyverse

### 3.1. Estructura del dataset


In [ ]:
# Estructura del dataset
cat(sprintf("📐 Dimensiones: %d filas × %d columnas\n", nrow(df), ncol(df)))
cat(sprintf("📐 Faltantes totales: %d\n\n", sum(is.na(df))))

# Tipos de variables
cat("🔍 Tipos de variables:\n")
tipos <- sapply(df, class)
print(table(unlist(tipos)))


In [ ]:
# Resumen estadístico de variables numéricas
df %>%
  select(where(is.numeric)) %>%
  summary()


### 3.2. Visualización de la variable respuesta


In [ ]:
# Distribución de Y con ggplot2
df_plot <- df %>%
  count(Y) %>%
  mutate(
    etiqueta = ifelse(Y == 0, "Pago regular (Y=0)", "Default (Y=1)"),
    porcentaje = n / sum(n)
  )

ggplot(df_plot, aes(x = etiqueta, y = n, fill = factor(Y))) +
  geom_col(width = 0.6, color = "white", linewidth = 1.5) +
  geom_text(aes(label = sprintf("%d\n(%.1f%%)", n, porcentaje * 100)),
            vjust = -0.4, fontface = "bold", color = COLOR_INDIGO, size = 4) +
  scale_fill_manual(values = c("0" = COLOR_EMERALD, "1" = COLOR_CARMIN),
                    guide = "none") +
  labs(title = "Distribución de la variable respuesta",
       subtitle = "German Credit Dataset · n = 1000",
       x = NULL, y = "Frecuencia absoluta") +
  ylim(0, max(df_plot$n) * 1.15)


### 3.3. Exploración de variables candidatas


In [ ]:
# Distribución de variables continuas por clase
library(patchwork)  # combinar gráficos

# Panel 1: Duración
p1 <- ggplot(df, aes(x = duration.in.month, fill = factor(Y))) +
  geom_density(alpha = 0.5, color = "white", linewidth = 0.8) +
  scale_fill_manual(values = c("0" = COLOR_EMERALD, "1" = COLOR_CARMIN),
                    labels = c("Pago regular", "Default"),
                    name = NULL) +
  labs(title = "Duración del crédito (meses)", x = "Meses", y = "Densidad")

# Panel 2: Monto en escala log
p2 <- ggplot(df, aes(x = credit.amount, fill = factor(Y))) +
  geom_density(alpha = 0.5, color = "white", linewidth = 0.8) +
  scale_x_log10() +
  scale_fill_manual(values = c("0" = COLOR_EMERALD, "1" = COLOR_CARMIN),
                    labels = c("Pago regular", "Default"),
                    name = NULL) +
  labs(title = "Monto del crédito (escala log)", x = "Monto", y = "Densidad")

# Panel 3: Edad
p3 <- ggplot(df, aes(x = age.in.years, fill = factor(Y))) +
  geom_density(alpha = 0.5, color = "white", linewidth = 0.8) +
  scale_fill_manual(values = c("0" = COLOR_EMERALD, "1" = COLOR_CARMIN),
                    labels = c("Pago regular", "Default"),
                    name = NULL) +
  labs(title = "Edad del solicitante", x = "Años", y = "Densidad")

# Panel 4: Tasa de default por historial
tasa_hist <- df %>%
  group_by(credit.history) %>%
  summarise(tasa = mean(Y), .groups = "drop") %>%
  arrange(desc(tasa))

p4 <- ggplot(tasa_hist, aes(x = tasa,
                            y = fct_reorder(credit.history, tasa))) +
  geom_col(fill = COLOR_VIOLET, width = 0.7) +
  geom_text(aes(label = sprintf("%.1f%%", tasa * 100)),
            hjust = -0.1, color = COLOR_INDIGO, fontface = "bold", size = 3.5) +
  geom_vline(xintercept = mean(df$Y), color = COLOR_GOLD,
             linetype = "dashed", linewidth = 1) +
  labs(title = "Tasa de default por historial",
       x = "Tasa de default", y = NULL) +
  xlim(0, max(tasa_hist$tasa) * 1.2)

# Combinar
(p1 + p2) / (p3 + p4)


### 📝 Interpretación exploratoria

Las visualizaciones revelan los mismos patrones que en el notebook de Python:

- **Duración**: los defaults (rojo) se desplazan ligeramente hacia valores más altos
- **Monto**: distribución log-normal con leve sesgo de defaults hacia montos altos
- **Edad**: efecto protector visible — defaults concentrados en franja 25-35
- **Historial crediticio**: **patrón contraintuitivo** — la categoría `critical` tiene la **menor** tasa de default (representa clientes conocidos del sistema), mientras que `no credits/all paid` y `all paid` presentan tasas elevadas

> 💡 Esta lectura contraintuitiva ilustra por qué el conocimiento de negocio es indispensable: las etiquetas no siempre significan lo que sugieren.


## 4. Diagnóstico de cardinalidad y agrupamiento preventivo

Replicamos el diagnóstico preventivo del notebook de Python: inspeccionar la frecuencia y la tasa de default por cada categoría, marcar las categorías raras (n < 30) y agruparlas en `other_minor` para prevenir la separación perfecta en el modelo.


In [ ]:
# Selección de variables del modelo
vars_continuas <- c("duration.in.month", "credit.amount", "age.in.years",
                    "installment.rate.in.percentage.of.disposable.income",
                    "present.residence.since")
vars_categoricas <- c("credit.history", "purpose", "savings.account.and.bonds")

cat(sprintf("📋 Variables continuas (%d):\n", length(vars_continuas)))
cat(sprintf("   %s\n\n", paste(vars_continuas, collapse = ", ")))
cat(sprintf("📋 Variables categóricas (%d):\n", length(vars_categoricas)))
cat(sprintf("   %s\n", paste(vars_categoricas, collapse = ", ")))


In [ ]:
# Diagnóstico de cardinalidad
UMBRAL_FRECUENCIA <- 30

cat("🔍 Diagnóstico de cardinalidad por variable categórica\n")
cat(sprintf("   Umbral de frecuencia mínima: %d casos\n\n", UMBRAL_FRECUENCIA))

for (var in vars_categoricas) {
  cat(sprintf("━━━ %s ━━━\n", var))

  tabla <- df %>%
    group_by(.data[[var]]) %>%
    summarise(
      `Y=0 (good)` = sum(Y == 0),
      `Y=1 (bad)`  = sum(Y == 1),
      Total        = n(),
      `Tasa default` = round(mean(Y), 3),
      .groups = "drop"
    ) %>%
    mutate(Diagnostico = case_when(
      Total < UMBRAL_FRECUENCIA & (`Y=1 (bad)` == 0 | `Y=0 (good)` == 0) ~ "❌ Sep + raro",
      Total < UMBRAL_FRECUENCIA ~ "⚠️  Raro (n<30)",
      `Y=1 (bad)` == 0 | `Y=0 (good)` == 0 ~ "❌ Sep. perfecta",
      TRUE ~ ""
    )) %>%
    rename(Categoria = 1)

  print(tabla, n = Inf)
  cat("\n")
}


In [ ]:
# Agrupamiento de categorías raras en 'other_minor'
df_agrup <- df

cat("🔧 Agrupando categorías con menos de 30 casos en 'other_minor':\n\n")

for (var in vars_categoricas) {
  freq <- table(df[[var]])
  categorias_raras <- names(freq[freq < UMBRAL_FRECUENCIA])

  if (length(categorias_raras) > 0) {
    cat(sprintf("   %s:\n", var))
    for (cat_rara in categorias_raras) {
      cat(sprintf("      • '%s': %d casos → agrupar\n", cat_rara, freq[cat_rara]))
    }
    # Reemplazo
    df_agrup[[var]] <- as.character(df[[var]])
    df_agrup[[var]][df_agrup[[var]] %in% categorias_raras] <- "other_minor"
    df_agrup[[var]] <- factor(df_agrup[[var]])

    n_other <- sum(df_agrup[[var]] == "other_minor")
    cat(sprintf("      → Nueva categoría 'other_minor' con %d casos\n\n", n_other))
  } else {
    cat(sprintf("   %s: ninguna categoría rara, no requiere agrupamiento.\n\n", var))
  }
}

# Reemplazamos df por la versión agrupada
df <- df_agrup
cat("✅ Variables categóricas agrupadas. Continuamos con la versión limpia.\n")


In [ ]:
# Verificación EPV post-agrupamiento
n_eventos <- sum(df$Y)
n_cat_dummies <- sum(sapply(vars_categoricas, function(v) length(unique(df[[v]])) - 1))
n_variables_total <- length(vars_continuas) + n_cat_dummies

EPV <- n_eventos / n_variables_total

cat("🔍 Diagnóstico EPV (Events Per Variable)\n")
cat(sprintf("   Eventos (Y=1):            %d\n", n_eventos))
cat(sprintf("   Variables (con dummies):  %d\n", n_variables_total))
cat(sprintf("   EPV:                      %.1f\n\n", EPV))

if (EPV >= 10) {
  cat(sprintf("   ✅ EPV = %.1f cumple holgadamente la regla de Peduzzi.\n", EPV))
} else if (EPV >= 5) {
  cat(sprintf("   ⚠️  EPV = %.1f entre 5 y 10: aceptable con regularización.\n", EPV))
} else {
  cat(sprintf("   ❌ EPV = %.1f < 5: alto riesgo de sobreajuste.\n", EPV))
}


## 5. Ajuste con `glm()` base — MV pura

A diferencia de sklearn (que aplica regularización L2 por defecto), `glm()` de R ajusta por **máxima verosimilitud pura**. Esta es la convención del análisis estadístico clásico y la que permite construir inferencia con errores estándar e intervalos de confianza válidos.

### Partición train/test estratificada con `scorecard::split_df()`

El paquete `scorecard` incluye su propia función de partición estratificada que respeta la proporción de la variable respuesta.


In [ ]:
# Preparación del dataset modelo
df_modelo <- df %>%
  select(all_of(c(vars_continuas, vars_categoricas)), Y) %>%
  mutate(across(all_of(vars_categoricas), as.factor))

# Partición train/test estratificada (70/30) usando scorecard
set.seed(RNG_SEED)
particion <- split_df(df_modelo, y = "Y", ratio = c(0.7, 0.3), seed = RNG_SEED)
train <- particion$train
test  <- particion$test

cat(sprintf("🎓 Train: %d filas — tasa default: %.2f%%\n",
            nrow(train), mean(train$Y) * 100))
cat(sprintf("🧪 Test : %d filas — tasa default: %.2f%%\n",
            nrow(test), mean(test$Y) * 100))


In [ ]:
# Ajuste del modelo con glm() base
formula_modelo <- as.formula(paste("Y ~", paste(c(vars_continuas, vars_categoricas),
                                                 collapse = " + ")))

cat("📋 Fórmula del modelo:\n")
print(formula_modelo)
cat("\n")

modelo_glm <- glm(formula_modelo,
                  data   = train,
                  family = binomial(link = "logit"))

# Diagnóstico de convergencia
cat("🔍 Diagnóstico del ajuste:\n\n")
cat(sprintf("   ✅ Convergencia: %s\n", modelo_glm$converged))
cat(sprintf("   ✅ Iteraciones:  %d\n", modelo_glm$iter))
cat(sprintf("   ✅ Log-verosimilitud: %.2f\n", logLik(modelo_glm)))
cat(sprintf("   ✅ AIC:               %.2f\n", AIC(modelo_glm)))

# Pseudo R² de McFadden
modelo_nulo <- glm(Y ~ 1, data = train, family = binomial())
pseudo_R2 <- 1 - (logLik(modelo_glm) / logLik(modelo_nulo))
cat(sprintf("   ✅ Pseudo R² (McFadden): %.4f\n", as.numeric(pseudo_R2)))

# Rango de coeficientes
beta_max <- max(abs(coef(modelo_glm)))
se_max <- max(summary(modelo_glm)$coefficients[, "Std. Error"])
cat(sprintf("\n   ✅ |β̂| máx = %.2f, SE máx = %.2f\n", beta_max, se_max))


In [ ]:
# Resumen clásico del modelo (estilo SAS/regulatorio)
summary(modelo_glm)


## 6. Interpretación de coeficientes y odds ratios

Construimos la **tabla profesional** que normalmente se reporta en un informe regulatorio: $\hat{\beta}$, SE, estadístico z, p-valor, OR e IC 95 %.

### Función `tidy()` de broom para salida ordenada


In [ ]:
# Tabla profesional con broom::tidy()
tabla_coefs <- modelo_glm %>%
  tidy(conf.int = TRUE, conf.level = 0.95) %>%
  mutate(
    OR        = exp(estimate),
    OR_inf    = exp(conf.low),
    OR_sup    = exp(conf.high),
    `Sig.`    = case_when(
      p.value < 0.001 ~ "***",
      p.value < 0.01  ~ "**",
      p.value < 0.05  ~ "*",
      p.value < 0.10  ~ ".",
      TRUE            ~ ""
    )
  ) %>%
  rename(
    Variable = term,
    `β̂`      = estimate,
    `SE(β̂)`  = std.error,
    `z`      = statistic,
    `p-valor` = p.value,
    `IC 95% inf` = conf.low,
    `IC 95% sup` = conf.high
  ) %>%
  select(Variable, `β̂`, `SE(β̂)`, z, `p-valor`,
         OR, OR_inf, OR_sup, Sig.) %>%
  mutate(across(where(is.numeric), ~round(., 4)))

cat("📊 Tabla profesional de coeficientes\n")
cat("   Códigos: *** p<0.001, ** p<0.01, * p<0.05, . p<0.10\n\n")
print(tabla_coefs, n = Inf)


### 6.1. Forest plot de odds ratios


In [ ]:
# Forest plot de odds ratios (excluyendo intercepto)
forest_data <- tabla_coefs %>%
  filter(Variable != "(Intercept)") %>%
  mutate(
    Variable_corta = str_trunc(Variable, 38),
    Variable_corta = fct_reorder(Variable_corta, OR),
    color_efecto = ifelse(OR > 1, "riesgo", "protector")
  )

ggplot(forest_data, aes(x = OR, y = Variable_corta, color = color_efecto)) +
  geom_vline(xintercept = 1, linetype = "dashed",
             color = COLOR_INDIGO, linewidth = 0.5) +
  geom_errorbarh(aes(xmin = OR_inf, xmax = OR_sup), height = 0,
                 linewidth = 1, alpha = 0.5) +
  geom_point(size = 3, alpha = 0.9) +
  scale_color_manual(values = c("riesgo" = COLOR_CARMIN,
                                "protector" = COLOR_EMERALD),
                     guide = "none") +
  scale_x_log10(breaks = c(0.05, 0.1, 0.2, 0.5, 1, 2, 5, 10)) +
  labs(title = "Odds ratios estimados con IC 95 %",
       subtitle = "Modelo: glm() — German Credit Dataset",
       x = "Odds Ratio (escala log)",
       y = NULL) +
  annotate("text", x = 0.06, y = 0.5,
           label = "← Reduce riesgo",
           color = COLOR_EMERALD, fontface = "bold", size = 3.5) +
  annotate("text", x = 8, y = 0.5,
           label = "Incrementa riesgo →",
           color = COLOR_CARMIN, fontface = "bold", size = 3.5)


### 📝 Lectura económica de coeficientes clave

A partir de la tabla y el forest plot, podemos elaborar la lectura económica:

**Variables continuas significativas** (interpretación por unidad original):

- **Duración del crédito**: cada mes adicional incrementa los odds de default en aproximadamente un 3 %.
- **Monto del crédito**: cada DM adicional incrementa los odds en una pequeña fracción; el efecto se acumula con el monto total.
- **Cuota mensual (installment rate)**: cuotas más pesadas como % del ingreso disponible incrementan el riesgo.

**Variables categóricas más relevantes**:

- **Historial crediticio**: las categorías distintas de `all paid` (la referencia) muestran odds ratios **menores que 1**, indicando reducción de riesgo. La categoría `critical/other existing credit` presenta el efecto protector más fuerte (clientes conocidos del sistema).
- **Propósito**: `education` y `new car` incrementan los odds; `used car` los reduce.
- **Ahorros**: tener saldos superiores a 1000 DM (`>=1000`) reduce drásticamente los odds.


## 7. Flujo industrial completo con el paquete `scorecard`

Aquí está la **gran ventaja de R sobre Python** en credit scoring: el paquete `scorecard` de Xie (2020) automatiza el flujo completo de la industria en cinco líneas:

1. **`woebin()`**: discretización supervisada óptima mediante árboles de decisión
2. **`woebin_ply()`**: aplicación de la transformación WoE
3. **`glm()`**: ajuste del modelo logístico sobre variables WoE
4. **`scorecard()`**: construcción de la scorecard con parámetros de Siddiqi
5. **`scorecard_ply()`**: aplicación a nuevos datos

### 7.1. Binning supervisado automático


In [ ]:
# Binning supervisado con scorecard::woebin()
# Internamente usa árboles de decisión para encontrar puntos de corte óptimos

cat("⏳ Ejecutando binning supervisado (puede tardar 10-20 segundos)...\n\n")

bins <- woebin(
  dt    = train,
  y     = "Y",
  x     = c(vars_continuas, vars_categoricas),
  method = "tree",       # método de partición por árboles
  positive = "1",        # clase positiva = default
  print_info = FALSE
)

cat(sprintf("✅ Binning completado para %d variables\n\n", length(bins)))

# Visualizar el binning de las variables más interesantes
cat("📋 Ejemplo: binning de 'duration.in.month'\n\n")
print(bins$duration.in.month)


In [ ]:
# Information Value (IV) por variable: ranking de poder predictivo
iv_ranking <- iv(train, y = "Y",
                 x = c(vars_continuas, vars_categoricas),
                 positive = "1") %>%
  arrange(desc(info_value)) %>%
  mutate(
    Interpretacion = case_when(
      info_value >= 0.50 ~ "🟢 Sospechoso (verificar)",
      info_value >= 0.30 ~ "🟢 Fuerte",
      info_value >= 0.10 ~ "🟡 Medio",
      info_value >= 0.02 ~ "🟠 Débil",
      TRUE               ~ "🔴 Irrelevante"
    )
  )

cat("📊 Information Value (IV) — Ranking de poder predictivo\n\n")
print(iv_ranking)


### 📝 Interpretación del Information Value

El IV es la métrica estándar de la industria para ranking de variables en credit scoring. Reglas heurísticas:

| Rango IV | Interpretación |
|----------|----------------|
| ≥ 0.50 | **Sospechoso** — verificar data leakage |
| 0.30 – 0.50 | **Fuerte** |
| 0.10 – 0.30 | **Medio** |
| 0.02 – 0.10 | **Débil** |
| < 0.02 | **Irrelevante** — descartar |

> 💡 Las variables con IV ≥ 0.30 deben revisarse: pueden ser excelentes predictores o reflejar fuga de información (data leakage) desde el target.


In [ ]:
# Aplicación de WoE a train y test
train_woe <- woebin_ply(train, bins, print_info = FALSE)
test_woe  <- woebin_ply(test,  bins, print_info = FALSE)

cat(sprintf("📐 Train WoE: %d × %d\n", nrow(train_woe), ncol(train_woe)))
cat(sprintf("📐 Test WoE:  %d × %d\n\n", nrow(test_woe), ncol(test_woe)))

cat("📋 Primeras filas de train_woe:\n")
head(train_woe, 3)


### 7.2. Ajuste del modelo logístico sobre variables WoE

Las variables transformadas con WoE tienen relación **lineal con los log-odds por construcción**. Esto satisface automáticamente el supuesto de linealidad del modelo logístico.


In [ ]:
# Modelo logístico sobre datos transformados con WoE
modelo_woe <- glm(Y ~ ., data = train_woe, family = binomial())

cat("🔍 Diagnóstico del modelo WoE:\n\n")
cat(sprintf("   Convergencia: %s\n", modelo_woe$converged))
cat(sprintf("   Iteraciones:  %d\n", modelo_woe$iter))
cat(sprintf("   AIC: %.2f (vs modelo glm original: %.2f)\n",
            AIC(modelo_woe), AIC(modelo_glm)))
cat(sprintf("   Pseudo R² (McFadden): %.4f\n",
            as.numeric(1 - (logLik(modelo_woe) / logLik(modelo_nulo)))))


In [ ]:
# Resumen del modelo WoE
summary(modelo_woe)


### 7.3. Construcción de la scorecard final

Aplicamos la formulación de Siddiqi con la configuración estándar de la industria:

- **PDO** = 20 (Points to Double the Odds)
- **points0** = 600 (Score_target)
- **odds0** = 1/19 (Odds_target: bad:good = 1:19, equivalente a 50:1 good:bad)


In [ ]:
# Construcción de la scorecard con la función scorecard()
card <- scorecard(
  bins      = bins,
  model     = modelo_woe,
  points0   = 600,    # Score_target
  odds0     = 1/19,   # Odds_target (1 bad cada 19 good = 50 good por cada bad)
  pdo       = 20,     # Points to Double the Odds
  basepoints_eq0 = FALSE
)

cat("📋 Scorecard generada para las siguientes variables:\n")
cat(paste(" •", names(card), collapse = "\n"), "\n\n")

# Ejemplo: scorecard de una variable
cat("📊 Sub-scores para 'duration.in.month':\n\n")
print(card$duration.in.month)


In [ ]:
# Aplicación de la scorecard a train y test
score_train <- scorecard_ply(train, card, print_step = 0)
score_test  <- scorecard_ply(test,  card, print_step = 0)

cat(sprintf("📐 Scores generados:\n"))
cat(sprintf("   Train: %d scores\n", nrow(score_train)))
cat(sprintf("   Test:  %d scores\n\n", nrow(score_test)))

# Resumen de los scores en test
cat("📊 Resumen de scores (test):\n")
print(summary(score_test$score))
cat(sprintf("\n   Desvío estándar: %.1f\n", sd(score_test$score)))


In [ ]:
# Distribución del score por clase
df_score_test <- data.frame(
  score = score_test$score,
  Y     = test$Y
)

# Panel 1: KDE
p_kde <- ggplot(df_score_test, aes(x = score, fill = factor(Y))) +
  geom_density(alpha = 0.5, color = "white", linewidth = 0.8) +
  geom_vline(xintercept = median(df_score_test$score),
             color = COLOR_INDIGO, linetype = "dashed", linewidth = 0.8) +
  scale_fill_manual(values = c("0" = COLOR_EMERALD, "1" = COLOR_CARMIN),
                    labels = c("Pago regular", "Default"),
                    name = NULL) +
  labs(title = "Distribución del score por clase",
       subtitle = sprintf("Mediana: %.0f", median(df_score_test$score)),
       x = "Score crediticio", y = "Densidad")

# Panel 2: Boxplot
df_score_test$Y_label <- factor(df_score_test$Y,
                                levels = c(0, 1),
                                labels = c("Pago regular", "Default"))
p_box <- ggplot(df_score_test, aes(x = Y_label, y = score, fill = Y_label)) +
  geom_boxplot(width = 0.5, alpha = 0.8, color = COLOR_INDIGO) +
  scale_fill_manual(values = c("Pago regular" = COLOR_EMERALD,
                               "Default" = COLOR_CARMIN), guide = "none") +
  labs(title = "Comparación por clase",
       x = NULL, y = "Score crediticio")

p_kde + p_box


## 8. Métricas de evaluación con `perf_eva()`

El paquete `scorecard` incluye `perf_eva()` que calcula todas las métricas estándar de la industria en una sola llamada y produce visualizaciones profesionales.


In [ ]:
# Predicciones de probabilidad sobre test
prob_test <- predict(modelo_woe, newdata = test_woe, type = "response")

# Evaluación completa con perf_eva
perf_eva_resultado <- perf_eva(
  pred    = prob_test,
  label   = test_woe$Y,
  title   = "test",
  binomial_metric = c("ks", "auc", "gini"),
  positive = "1",
  show_plot = c("ks", "roc")
)

# Extracción robusta de las métricas (la estructura puede variar entre versiones)
metricas_test <- perf_eva_resultado$binomial_metric$test

cat("📊 Métricas de performance sobre conjunto de testeo:\n\n")
print(metricas_test)

# Guardamos en variables individuales para uso posterior
auc_test  <- as.numeric(metricas_test$AUC)
ks_test   <- as.numeric(metricas_test$KS)
gini_test <- as.numeric(metricas_test$Gini)


In [ ]:
# Tabla resumen con umbrales industriales
metricas_resumen <- data.frame(
  Metrica = c("AUC-ROC", "KS Kolmogorov-Smirnov", "Gini"),
  Valor   = c(auc_test, ks_test, gini_test),
  Umbral_minimo = c(0.70, 0.30, 0.40)
) %>%
  mutate(
    Evaluacion = case_when(
      Metrica == "AUC-ROC" & Valor >= 0.80 ~ "✅ Excelente",
      Metrica == "AUC-ROC" & Valor >= 0.75 ~ "✅ Bueno",
      Metrica == "AUC-ROC" & Valor >= 0.70 ~ "✅ Aceptable",
      Metrica == "AUC-ROC" & Valor >= 0.65 ~ "⚠️  Débil",
      Metrica == "AUC-ROC" ~ "❌ Insuficiente",
      Metrica == "KS Kolmogorov-Smirnov" & Valor >= 0.50 ~ "✅ Excelente",
      Metrica == "KS Kolmogorov-Smirnov" & Valor >= 0.40 ~ "✅ Bueno",
      Metrica == "KS Kolmogorov-Smirnov" & Valor >= 0.30 ~ "✅ Aceptable",
      Metrica == "KS Kolmogorov-Smirnov" ~ "❌ Insuficiente",
      Metrica == "Gini" & Valor >= 0.60 ~ "✅ Excelente",
      Metrica == "Gini" & Valor >= 0.50 ~ "✅ Bueno",
      Metrica == "Gini" & Valor >= 0.40 ~ "✅ Aceptable",
      TRUE ~ "❌ Insuficiente"
    ),
    Valor = round(Valor, 4)
  )

cat("📊 Resumen evaluación industrial:\n\n")
print(metricas_resumen)


### 8.1. Population Stability Index (PSI) train vs test

El **PSI** mide si la distribución del score en test difiere significativamente de la del train. Es la métrica clave para monitorear estabilidad temporal del modelo en producción.

| Rango PSI | Interpretación |
|-----------|----------------|
| < 0.10 | Sin cambios significativos |
| 0.10 – 0.25 | Cambios menores: monitorear |
| > 0.25 | Cambios mayores: revisar/recalibrar |


In [ ]:
# PSI del score entre train y test
psi_score <- perf_psi(
  score = list(train = score_train, test = score_test),
  label = list(train = train$Y,     test = test$Y),
  show_plot = FALSE
)

cat("📊 PSI del score (train vs test):\n\n")
print(psi_score$psi)

# Extracción robusta del valor PSI
# La columna puede llamarse 'PSI' o 'psi' según versión del paquete
psi_df <- as.data.frame(psi_score$psi)
col_psi <- grep("^psi$", names(psi_df), ignore.case = TRUE, value = TRUE)
if (length(col_psi) == 0) col_psi <- "psi"  # fallback
psi_valor <- as.numeric(psi_df[[col_psi]][1])

cat("\n📝 Interpretación:\n")
if (psi_valor < 0.10) {
  cat(sprintf("   ✅ PSI = %.4f: distribuciones estables\n", psi_valor))
} else if (psi_valor < 0.25) {
  cat(sprintf("   ⚠️  PSI = %.4f: cambios menores, monitorear\n", psi_valor))
} else {
  cat(sprintf("   ❌ PSI = %.4f: cambios mayores, recalibrar\n", psi_valor))
}


## 9. Diagnóstico de supuestos

### 9.1. Multicolinealidad mediante VIF

El paquete `car` provee `vif()` que calcula el Variance Inflation Factor directamente sobre un objeto `glm`.


In [ ]:
# VIF sobre el modelo WoE
vif_resultado <- vif(modelo_woe)

vif_df <- data.frame(
  Variable = names(vif_resultado),
  VIF = as.numeric(vif_resultado)
) %>%
  arrange(desc(VIF)) %>%
  mutate(
    Diagnostico = case_when(
      VIF < 5  ~ "✅ Leve o ausente",
      VIF < 10 ~ "⚠️  Moderada",
      TRUE     ~ "❌ Severa"
    ),
    VIF = round(VIF, 2)
  )

cat("📊 Diagnóstico de multicolinealidad (VIF)\n\n")
cat(sprintf("   Variables con VIF severo (≥ 10):   %d\n",
            sum(vif_df$VIF >= 10)))
cat(sprintf("   Variables con VIF moderado (5-10): %d\n",
            sum(vif_df$VIF >= 5 & vif_df$VIF < 10)))
cat(sprintf("   Variables sin problema (< 5):      %d\n\n",
            sum(vif_df$VIF < 5)))

print(vif_df, row.names = FALSE)


### 9.2. Linealidad en los logits (modelo glm original)

Sobre el modelo WoE no aplica este diagnóstico porque las variables transformadas son lineales por construcción. Aplicamos el diagnóstico al modelo `glm()` original con variables continuas crudas.


In [ ]:
# Función de diagnóstico de linealidad por deciles
diagnostico_linealidad <- function(datos, variable, target = "Y", n_bins = 10) {
  datos %>%
    mutate(bin = ntile(.data[[variable]], n_bins)) %>%
    group_by(bin) %>%
    summarise(
      centro = mean(.data[[variable]]),
      tasa_default = mean(.data[[target]]),
      .groups = "drop"
    ) %>%
    mutate(
      p_clip = pmin(pmax(tasa_default, 0.001), 0.999),
      logit_empirico = log(p_clip / (1 - p_clip))
    )
}

# Crear gráficos para cada variable continua
plots_linealidad <- lapply(vars_continuas, function(var) {
  res <- diagnostico_linealidad(train, var)

  # Línea de regresión simple como referencia
  modelo_lineal <- lm(logit_empirico ~ centro, data = res)
  x_line <- seq(min(res$centro), max(res$centro), length.out = 100)
  y_line <- predict(modelo_lineal, newdata = data.frame(centro = x_line))
  ref_df <- data.frame(centro = x_line, logit_referencia = y_line)

  ggplot(res, aes(x = centro, y = logit_empirico)) +
    geom_line(data = ref_df, aes(y = logit_referencia),
              color = COLOR_GOLD, linetype = "dashed", linewidth = 1) +
    geom_point(size = 3, color = COLOR_VIOLET) +
    labs(title = var, x = var, y = "Logit empírico") +
    theme(plot.title = element_text(size = 9))
})

# Combinar todos los gráficos
wrap_plots(plots_linealidad, ncol = length(vars_continuas))


### 📝 Lectura del diagnóstico

Si los puntos (deciles) se alinean razonablemente con la recta de referencia dorada, el supuesto de linealidad se cumple. Curvaturas sistemáticas sugieren transformar la variable o usar binning con WoE.

> 💡 **Esta es justamente la ventaja del flujo `scorecard`**: el WoE linealiza automáticamente las relaciones, eliminando la necesidad de transformaciones manuales.


## 10. Síntesis y comparación R vs Python

### 10.1. Resumen ejecutivo del modelo


In [ ]:
# Tabla ejecutiva del modelo R
resumen_final <- data.frame(
  Metrica = c(
    "Tamaño muestra train",
    "Tamaño muestra test",
    "Variables en modelo WoE",
    "EPV (post-agrupamiento)",
    "AIC modelo WoE",
    "Pseudo R² McFadden (WoE)",
    "AUC-ROC (test)",
    "KS (test)",
    "Gini (test)",
    "PSI (train vs test)",
    "Score mínimo (test)",
    "Score máximo (test)",
    "Score mediana (test)"
  ),
  Valor = c(
    nrow(train),
    nrow(test),
    length(coef(modelo_woe)) - 1,
    sprintf("%.1f", EPV),
    sprintf("%.2f", AIC(modelo_woe)),
    sprintf("%.4f", as.numeric(1 - (logLik(modelo_woe) / logLik(modelo_nulo)))),
    sprintf("%.4f", auc_test),
    sprintf("%.4f", ks_test),
    sprintf("%.4f", gini_test),
    sprintf("%.4f", psi_valor),
    sprintf("%.1f", min(score_test$score)),
    sprintf("%.1f", max(score_test$score)),
    sprintf("%.1f", median(score_test$score))
  )
)

cat("📊 Resumen ejecutivo del modelo (R)\n\n")
print(resumen_final, row.names = FALSE)


### 10.2. Comparativa R vs Python sobre el mismo dataset

| Aspecto | Python (sklearn + statsmodels) | R (`scorecard` + glm) |
|---------|-------------------------------|----------------------|
| **Líneas de código** | ~150 líneas en notebook | ~80 líneas en notebook |
| **Discretización WoE** | Manual con `optbinning` (opcional) | **Automática con `woebin()`** |
| **Selección por IV** | Manual | **Integrada (`iv()`)** |
| **Forest plot** | Manual con matplotlib | Manual con ggplot2 |
| **AUC, KS, Gini juntos** | Cálculo separado | **`perf_eva()` en una llamada** |
| **Scorecard final** | Manual con `prob_a_score()` | **`scorecard()` integrada** |
| **PSI train vs test** | Manual | **`perf_psi()` integrada** |
| **Salida `summary()`** | statsmodels (z-test) | glm() (estilo SAS) |
| **Convención regulatoria** | Requiere statsmodels para inferencia | **Listo para regulador** |

### 10.3. ¿Cuándo elegir cada lenguaje?

**Elegir R cuando:**
- El proyecto tiene componente regulatorio fuerte (Basilea, BCRA)
- El equipo viene de SAS y busca lenguaje estadístico similar
- Se construyen scorecards tradicionales con WoE
- Se requiere reportería estadística clásica

**Elegir Python cuando:**
- El proyecto integra credit scoring con otras tareas de ML
- Hay pipelines productivos que conectan con APIs, microservicios
- Se experimenta con modelos más allá de logística (XGBoost, redes)
- El equipo es predominantemente de data science / ML engineering

> 💡 **Recomendación profesional**: en credit scoring contemporáneo, dominar ambos lenguajes es la norma. R suele dominar en bancos tradicionales argentinos; Python en fintechs y bancos digitales.


## 📋 Entregable parcial (recordatorio)

Sobre la base de este notebook (o el de Python, a elección), entregar antes del inicio de la Clase 6:

1. ✅ Ajuste de un modelo logístico con ≥ 5 variables explicativas (justificadas)
2. ✅ Reporte de coeficientes con odds ratio e IC 95 %
3. ✅ Construcción de scorecard con PDO=20, anclaje 600/50:1
4. ✅ Cálculo de AUC, KS, Gini sobre testeo
5. ✅ Interpretación económica de tres coeficientes en lenguaje gerencial

### 🎯 Criterios de evaluación

| Dimensión | Peso |
|-----------|------|
| Corrección técnica del código | 30 % |
| Interpretación económica de coeficientes | 30 % |
| Construcción adecuada de la scorecard | 20 % |
| Calidad del reporte final | 20 % |

---

### 🔮 Anticipación: Clase 6 (jueves 14 de mayo)

- **Árboles de decisión CART**: criterios de partición (Gini, entropía), poda
- **Random Forest**: bagging, doble aleatorización, importancia de variables
- **Comparativa empírica** con el modelo logístico de hoy sobre el mismo dataset

---

<center>

**Dr. Darío Ezequiel Díaz**
*Aplicaciones de Minería de Datos a Economía y Finanzas*
Maestría en Minería de Datos · UTN Facultad Regional Rosario · 2026

</center>
